# FPSA-Prime verification and controlled Maze experiment

This notebook validates the numerical implementation first, then optionally runs the controlled Maze comparison. The full experiment keeps strict fixed-point checks enabled. If a batch misses the initial numerical cap, the launcher retries the same fixed-point equation with a larger cap and records the realized cost. It never scores an unconverged state as an equilibrium.


In [ ]:
REPO_URL = "https://github.com/mrinal18/fpsa.git"
BRANCH = "fpsa-prime-v0"

QUICK = True
RUN_TINY_TRAINING = True

RUN_FULL_BENCHMARK = False
FULL_SEEDS = [0]              # expand after one clean run
FULL_STEPS = 1200
TRAIN_INITIAL_CAP = 24
TRAIN_RETRY_CEILING = 96      # retry schedule: 24 -> 48 -> 96
EVAL_INITIAL_CAP = 64
EVAL_RETRY_CEILING = 256      # retry schedule: 64 -> 128 -> 256


## 1. Checkout the latest branch and install dependencies


In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
target = Path("/content/fpsa") if IN_COLAB else Path.cwd()
if not (target / ".git").exists():
    if target.exists() and target != Path.cwd():
        subprocess.run(["rm", "-rf", str(target)], check=True)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(target)], check=True)
else:
    os.chdir(target)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
os.chdir(target)
sys.path.insert(0, str(target))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "pandas", "matplotlib"], check=True)
import pandas as pd
import matplotlib.pyplot as plt
import torch
print({
    "git_sha": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})


## 2. Run the portable numerical verification harness


In [ ]:
for module_name in list(sys.modules):
    if module_name.startswith("experiments.fpsa_prime.verification"):
        del sys.modules[module_name]
from experiments.fpsa_prime.verification import run_verification
summary = run_verification(
    output_dir="results/verification_colab",
    device="auto",
    quick=QUICK,
    run_tests=True,
    run_training=RUN_TINY_TRAINING,
)
print(json.dumps({
    "tests": summary["tests"].splitlines()[-1],
    "parameter_gap_percent": 100 * summary["hero_block_parameter_gap_fraction"],
    "gmres_diagnostic": summary["gmres_diagnostic"],
    "plots": summary["plots"],
}, indent=2))


In [ ]:
from IPython.display import Image, display
display(pd.DataFrame(summary["parameter_counts"]))
display(pd.DataFrame(summary["gmres"]))
display(pd.DataFrame(summary["stability_power_sweep"]))
if summary["tiny_training"] is not None:
    display(pd.DataFrame(summary["tiny_training"]))
for plot_name in summary["plots"]:
    print(plot_name)
    display(Image(filename=f"results/verification_colab/{plot_name}"))


## 3. Optional full controlled Maze experiment

The launcher streams and saves the child-process log. A nonzero exit now prints the final 120 log lines and writes a `.failure.json` plus `.failure.pt` checkpoint. Strict convergence remains enabled. The initial caps are charged only when used; cap retries and realized NFE are recorded in the result.


In [ ]:
def run_streaming(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("\n$", shlex.join(command), flush=True)
    lines = []
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()
            lines.append(line.rstrip())
            if len(lines) > 120:
                lines.pop(0)
        return_code = process.wait()
    if return_code != 0:
        print("\n--- final child-process log lines ---")
        print("\n".join(lines))
        failure_files = sorted(log_path.parent.glob("*.failure.*"))
        if failure_files:
            print("Failure diagnostics:", [str(path) for path in failure_files])
        raise RuntimeError(
            f"Command failed with exit code {return_code}. Full log: {log_path}"
        )

if RUN_FULL_BENCHMARK:
    output_dir = Path("results/controlled_colab_adaptive_strict")
    output_dir.mkdir(parents=True, exist_ok=True)
    shared = [
        "--task", "maze",
        "--maze_size", "7",
        "--extra_sizes", "9", "11",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--steps", str(FULL_STEPS),
        "--batch_size", "32",
        "--lr", "3e-3",
        "--hidden", "128",
        "--heads", "4",
        "--max_iter", str(TRAIN_INITIAL_CAP),
        "--max_iter_eval", str(EVAL_INITIAL_CAP),
        "--train_max_iter_ceiling", str(TRAIN_RETRY_CEILING),
        "--eval_max_iter_ceiling", str(EVAL_RETRY_CEILING),
        "--fp_tol", "1e-4",
        "--eval_every", "100",
    ]
    for seed in FULL_SEEDS:
        prime_output = output_dir / f"prime_maze7_s{seed}.json"
        block_output = output_dir / f"deq_maze7_s{seed}.json"
        prime_command = [
            sys.executable, "-m", "experiments.fpsa_prime.controlled_compare",
            "--family", "prime", "--arch", "fpsa_prime",
            *shared, "--seed", str(seed),
            "--stability_power_steps", "4",
            "--output", str(prime_output),
            "--save_model", str(prime_output.with_suffix(".pt")),
        ]
        block_command = [
            sys.executable, "-m", "experiments.fpsa_prime.controlled_compare",
            "--family", "block", "--arch", "deq_block",
            *shared, "--seed", str(seed),
            "--output", str(block_output),
            "--save_model", str(block_output.with_suffix(".pt")),
        ]
        run_streaming(prime_command, output_dir / f"prime_maze7_s{seed}.log")
        run_streaming(block_command, output_dir / f"deq_maze7_s{seed}.log")

    rows = []
    histories = []
    for path in sorted(output_dir.glob("*.json")):
        if path.name.endswith(".failure.json"):
            continue
        result = json.loads(path.read_text())
        final = result["final"]
        rows.append({
            "family": result["family"],
            "arch": result["arch"],
            "seed": result["seed"],
            "params": result["params"],
            "maze7_em": final["exact_match"],
            "maze9_em": result["extra"]["size9"]["exact_match"],
            "maze11_em": result["extra"]["size11"]["exact_match"],
            "mean_nfe": final["mean_function_evals"],
            "p90_iterations": final.get("p90_sample_iterations"),
            "p99_iterations": final.get("p99_sample_iterations"),
            "max_iterations": final.get("max_sample_iterations"),
            "max_solver_cap": final.get("max_solver_cap"),
            "cap_retry_batches": final.get("batches_with_cap_escalation"),
            "max_residual": final.get("max_residual"),
            "elapsed_seconds": result["elapsed_seconds"],
        })
        for record in result["history"]:
            histories.append({
                "family": result["family"],
                "arch": result["arch"],
                "seed": result["seed"],
                **record,
            })
    full_results = pd.DataFrame(rows)
    history_table = pd.DataFrame(histories)
    display(full_results)
    if not full_results.empty:
        display(full_results.groupby(["family", "arch"]).agg(["mean", "std"]))
    if not history_table.empty:
        plt.figure(figsize=(8, 4.5))
        for (family, arch, seed), group in history_table.groupby(["family", "arch", "seed"]):
            plt.plot(group.step, group.exact_match, marker="o", label=f"{family}/{arch}/s{seed}")
        plt.xlabel("Training step")
        plt.ylabel("Maze-7 exact match (%)")
        plt.title("Controlled training trajectory")
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Full benchmark disabled. Set RUN_FULL_BENCHMARK=True after verification passes.")


## Interpretation rules

- `--allow_nonconvergence` must remain off.
- A cap retry changes only the numerical work used to solve the same equation; inspect `mean_nfe`, `max_solver_cap`, and retry counts.
- A failure at the final ceiling is a real convergence failure and produces diagnostic files.
- Start with one seed and inspect forward/adjoint residuals before expanding to three seeds.
